# Financial Transaction Data Cleaning Pipeline

**Objective:** To clean, standardize, and engineer features for a dataset of messy financial transactions for fraud detection analysis.

**Portfolio Focus:** Python, Pandas, Feature Engineering, Data Cleaning Pipelines.

In [33]:
import pandas as pd
import numpy as np
import datetime

# Define local paths relative to the notebook's location
input_filename = input('Enter your raw filename (e.g., financial_transactions_raw.csv): ') 
output_filename = "cleaned_" + input_filename

### Raw Data Analysis & Identified Issues

* **Inconsistent Timestamps:** The `timestamp` column contains a messy mixture of formats requiring normalization and explicit casting to `datetime64`.
* **Missing Temporal Features:** Machine learning models need time-based patterns. We must engineer `hour_of_day`, `is_weekend`, and `days_since_last_txn`.
* **Inconsistent Entity Names:** The `merchant` column suffers from severe data entry variations and differing legal suffixes.
* **Case Inconsistencies:** The `txn_type` column contains duplicate categories differing only by capitalization.
* **Extreme Outliers:** The `amount_inr` column has extreme maximums. We need to apply Winsorization (capping at the 99th percentile) specific to each transaction type.
* **Risk Flagging & Redundancy:** The `card_issue_country` column contains exactly 1 unique value ('IN'). Before dropping it, it must be cross-referenced with `txn_country` to create a `high_risk` flag.

In [39]:
# Ingestion: Automatically handling trailing spaces after delimiters
df = pd.read_csv(input_filename, skipinitialspace=True)

# Quick inspection of the raw data shape and initial rows
print(f"Initial shape: {df.shape}")
df.head()

Initial shape: (350, 9)


,txn_id,customer_id,timestamp,merchant,amount_inr,txn_type,txn_country,card_issue_country,fraud_label
0,TXN000001,CUST0022,45272,Amazon,48933.84,net_banking,GB,IN,0
1,TXN000002,CUST0023,2024-07-18T04:54:00,AMAZON INC,4631376.06,UPI,IN,IN,0
2,TXN000003,CUST0057,45294,Paytm,24317.99,net_banking,IN,IN,0
3,TXN000004,CUST0015,2024-09-16T22:54:00,Amazon,4793.23,Card,IN,IN,0
4,TXN000005,CUST0020,1710115200,Flipkart,42270.66,upi,GB,IN,0


In [35]:
# 1. Normalize Timestamps (Handling Excel, Unix, and ISO formats)
def standardize_timestamp(date_val):
    if pd.isna(date_val):
        return pd.NaT
    date_str = str(date_val).strip()
    if len(date_str) == 10 and date_str.isdigit():  # Unix
        return pd.to_datetime(int(date_str), unit='s')
    elif len(date_str) <= 5 and date_str.isdigit():  # Excel Serial (fix: <= instead of == to catch 4-digit serials)
        return pd.to_datetime('1899-12-30') + pd.to_timedelta(int(date_str), unit='D')
    else:
        try:
            return pd.to_datetime(date_str, format='mixed', errors='coerce')
        except ValueError:
            return pd.NaT

df['timestamp'] = pd.to_datetime(df['timestamp'].apply(standardize_timestamp))  # fix: chained into one line, ensures datetime64 dtype

# Engineered Features (Temporal Data)
df['hour_of_day'] = df['timestamp'].dt.hour
df['is_weekend'] = df['timestamp'].dt.dayofweek >= 5
# Sort to calculate days since last transaction accurately
df = df.sort_values(['customer_id', 'timestamp'])
df['days_since_last_txn'] = df.groupby('customer_id')['timestamp'].diff().dt.days.fillna(0)

# 2. Text Standardization (Merchant Names)
df['merchant'] = df['merchant'].str.lower().str.strip()
suffixes = [' pvt ltd', ' inc', ' ltd', ' limited']  # fix: ' pvt ltd' moved first so it's caught before ' ltd'
for suffix in suffixes:
    df['merchant'] = df['merchant'].str.replace(suffix, '', regex=False)
df['merchant'] = df['merchant'].str.strip()  # fix: strip again after suffix removal to remove leftover spaces
merchant_mapping = {
    'big basket': 'bigbasket',
    'uber india': 'uber',
    'swiggy india pvt': 'swiggy',
    'swiggy india': 'swiggy',
    'flipkart pvt': 'flipkart'
}
df['merchant'] = df['merchant'].replace(merchant_mapping)

# 3. Transaction Type Casing
df['txn_type'] = df['txn_type'].str.lower().str.strip()
df['txn_type'] = df['txn_type'].str.replace('netbanking', 'net_banking', regex=False)

# Winsorization (Capping Outliers per txn_type) — runs AFTER txn_type is normalized above
for txn in df['txn_type'].unique():
    mask = df['txn_type'] == txn
    cap = df.loc[mask, 'amount_inr'].quantile(0.99)
    df.loc[mask, 'amount_inr'] = df.loc[mask, 'amount_inr'].clip(upper=cap)

# Risk Flag Generation (BEFORE dropping card_issue_country)
df['high_risk'] = (df['txn_country'] != df['card_issue_country']).astype(int)

# Dimensionality Reduction (Dropping the Zero-Variance column)
if 'card_issue_country' in df.columns and len(df['card_issue_country'].unique()) == 1:
    df = df.drop('card_issue_country', axis=1, errors='ignore')

print(f"Remaining missing values: {df.isnull().sum().sum()}")
print(f"Final dataset shape: {df.shape}")

Remaining missing values: 0
Final dataset shape: (350, 12)


In [36]:
# Export the finalized, clean dataframe
df.to_csv(output_filename, index=False)
print(f"Data cleaning complete. Cleaned file saved securely at: {output_filename}")

Data cleaning complete. Cleaned file saved securely at: cleaned_financial_transactions_raw.csv
